In [ ]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
from datetime import datetime, timedelta


le = joblib.load('models/label_encoder.pkl')
scaler_X = joblib.load('models/scaler_X.pkl')
scaler_y = joblib.load('models/scaler_y.pkl')
meta_model = load_model('models/Neurostack_MetaLearner.h5', compile=False)


lr = joblib.load('models/linear_regression_model.pkl')
rf = joblib.load('models/random_forest_model.pkl')
xgb = joblib.load('models/xgboost_model.pkl')
lstm = load_model('models/BiLSTM_model.h5', compile=False)
gru = load_model('models/GRU_model.h5', compile=False)

print("Engine Resources Loaded Successfully.")

✅ Engine Resources Loaded Successfully.


In [ ]:
def run_inventory_engine(brand_name, current_stock, lead_time, usd_rate):

    brand_code = le.transform([brand_name])[0]

    input_data = np.array([[current_stock, 0, current_stock, 0, lead_time, usd_rate, brand_code]])
    input_scaled = scaler_X.transform(input_data)
    

    p_lr = lr.predict(input_scaled).ravel()
    p_rf = rf.predict(input_scaled).ravel()
    p_xgb = xgb.predict(input_scaled).ravel()
    input_dl = input_scaled.reshape((1, 1, 7))
    p_lstm = lstm.predict(input_dl, verbose=0).ravel()
    p_gru = gru.predict(input_dl, verbose=0).ravel()
    

    meta_in = np.array([[p_lr[0], p_rf[0], p_xgb[0], p_lstm[0], p_gru[0]]])
    pred_scaled = meta_model.predict(meta_in, verbose=0)
    predicted_demand = float(scaler_y.inverse_transform(pred_scaled)[0][0])
    

    daily_demand = predicted_demand / 30
    
    days_to_exhaust = current_stock / daily_demand if daily_demand > 0 else 365
    exhaustion_date = datetime.now() + timedelta(days=days_to_exhaust)
    

    risk_status = "High Risk" if days_to_exhaust <= lead_time else "Low Risk"
    risk_percentage = min(100, (lead_time / days_to_exhaust) * 100) if days_to_exhaust > 0 else 100


    suggested_order = max(0, predicted_demand - current_stock)
    

    buffer_note = "Ensuring 6-month safety buffer."
    
    return {
        "Brand": brand_name,
        "Predicted_Monthly_Demand": round(predicted_demand, 2),
        "Exhaustion_Date": exhaustion_date.strftime('%Y-%m-%d'),
        "Risk_Level": risk_status,
        "Risk_Percentage": f"{round(risk_percentage, 2)}%",
        "Suggested_Order_Qty": round(suggested_order, 2),
        "Note": buffer_note
    }


results = run_inventory_engine("Glilcomet", current_stock=200, lead_time=14, usd_rate=325.0)
print("\n--- ENGINE REPORT ---")
for key, value in results.items():
    print(f"{key}: {value}")

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



--- ENGINE REPORT ---
Brand: Glilcomet
Predicted_Monthly_Demand: 119.87
Exhaustion_Date: 2026-04-06
Risk_Level: Low Risk
Risk_Percentage: 27.97%
Suggested_Order_Qty: 0
Note: Ensuring 6-month safety buffer.
